# 01 — Bronze Layer: Raw Ingestion

Copies all 8 TPC-H tables from `samples.tpch` into `group1_finance.bronze` as-is. 

Table-level comments record the source and ingestion timestamp as metadata.

Prerequisite: run `00_setup` first.

In [0]:
# Configuration
CATALOG = "group1_finance"
SCHEMA = "bronze"

SOURCE_CATALOG = "samples"
SOURCE_SCHEMA  = "tpch"

TPCH_TABLES = [
    "customer",
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]

print(f"Target  : {CATALOG}.{SCHEMA}")
print(f"Source  : {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Tables  : {len(TPCH_TABLES)}")

Target  : group1_finance.bronze
Source  : samples.tpch
Tables  : 8


In [0]:
# Copy all TPC-H tables as-is into the bronze schema.
# CREATE OR REPLACE TABLE makes this idempotent (safe to re-run).
# No transformations — just a straight SELECT * with table-level metadata comments.

from pyspark.sql.functions import current_timestamp

ingested_at = spark.sql("SELECT current_timestamp() AS ts").collect()[0]["ts"]

for table in TPCH_TABLES:
    target = f"{CATALOG}.{SCHEMA}.{table}"
    source = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table}"

    spark.sql(f"CREATE OR REPLACE TABLE {target} AS SELECT * FROM {source}")

    spark.sql(
        f"""COMMENT ON TABLE {target} IS
        'Raw copy of {source}. Ingested: {ingested_at}'"""
    )

    row_count = spark.table(target).count()
    print(f"  {target:40s}  {row_count:>10,} rows")

print(f"\nAll {len(TPCH_TABLES)} tables copied to {CATALOG}.{SCHEMA}")

  group1_finance.bronze.customer               750,000 rows
  group1_finance.bronze.lineitem            29,999,795 rows
  group1_finance.bronze.nation                      25 rows
  group1_finance.bronze.orders               7,500,000 rows
  group1_finance.bronze.part                 1,000,000 rows
  group1_finance.bronze.partsupp             4,000,000 rows
  group1_finance.bronze.region                       5 rows
  group1_finance.bronze.supplier                50,000 rows

All 8 tables copied to group1_finance.bronze


In [0]:
# Verify: compare row counts between source and bronze
print(f"{'Table':<15s} {'Source':>10s} {'Bronze':>10s} {'Match':>6s}")
print("-" * 45)

all_match = True
for table in TPCH_TABLES:
    src_count = spark.table(f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table}").count()
    brz_count = spark.table(f"{CATALOG}.{SCHEMA}.{table}").count()
    match = "  match" if src_count == brz_count else "  mismatch"
    if src_count != brz_count:
        all_match = False
    print(f"{table:<15s} {src_count:>10,} {brz_count:>10,} {match}")

print("-" * 45)
print("All tables match" if all_match else "Mismatch detected")

Table               Source     Bronze  Match
---------------------------------------------
customer           750,000    750,000   match
lineitem        29,999,795 29,999,795   match
nation                  25         25   match
orders           7,500,000  7,500,000   match
part             1,000,000  1,000,000   match
partsupp         4,000,000  4,000,000   match
region                   5          5   match
supplier            50,000     50,000   match
---------------------------------------------
All tables match
